# Tutorial: How to Read `.bag` Data

In [ ]:
import pyrealsense2 as rs
import os
import numpy as np
import cv2
import gc

## Definitions

When you attempt to record videos using an Intel Realsense camera (e.g. D435i RGB-D camera, L515 LiDAR camera), the output of that operation is usually a `.bag` file. Realsense doesn't distinguish between cameras and `.bag` files; they are, in Intel's eyes, **essentially the same thing**. So consider your `.bag` files as "instances" of your camera.

This `.bag` data contains one or several "streams" of data. Each stream represents some element of the recording - i.e. the depth video, or the RGB video. In essence, a single `.bag` file can contain multiple streams.

Each stream in of itself has a specific type, format, resolution, FPS, etc. These are things you MUST note down when you are recording your footage. The `.bag` data doesn't contain any of this information, as far as I am aware.

## Goal of this Notebook

If we're given a `.bag` file to process, then it's expected that we want to extract a video per stream contained within that `.bag` file. We need to be _careful_ about this, as there are nuances to how `.bag` files work that cannot be ignored.

We have an example `.bag` file: `./samples_ignore/capstone/20251116_155458.bag`.

In [78]:
_EXAMPLE_BAG = '20251116_161222'
_FILE_DIR = '../samples_ignore/capstone/'

_INPUT_FILE = os.path.join(_FILE_DIR, _EXAMPLE_BAG+".bag")
_OUTPUT_FILE = os.path.join(_FILE_DIR, _EXAMPLE_BAG+".mp4")

### Common "Configuration" Class

Let's use a common class we can use EVERYWHERE to define things such as the width and height of a stream (i.e. the resolution).

In [ ]:
class _CONFIG_:
    def __init__(self, name:str, w:int, h:int, fps:int):
        self.name = name
        self.width = w
        self.height = h
        self.fps = fps

# In the case of an Intel Realsense stream:
class _INTEL_CONFIG_(_CONFIG_):
    def __init__(self, name:str, w:int, h:int, fps:int, type:rs.stream, format:rs.format, repeat:bool=False):
        super().__init__(name, w, h, fps)
        self.type = type
        self.format = format
        self.repeat = repeat

# In the case of an OpenCV-based video output:
class _OUTPUT_CONFIG_(_CONFIG_):
    def __init__(self, name:str, w:int, h:int, fps:int, interpolator=cv2.INTER_CUBIC, record:bool=True, record_filename:str=None, preview:bool=False):
        super().__init__(name, w, h, fps)
        self.interpolator = interpolator
        self.record = record
        self.preview = preview
        self.record_filename = name+".mp4" if record_filename is None else record_filename+".mp4"

        # if neither record or preview are set to True, will auto-default to record=true, preview=false
        if not self.record and not self.preview:
            print("ERROR: OUTPUT CONFIG CANNOT BE NEITHER RECORD OR PREVIEW. DEFAULTING TO RECORDING")
            self.record = True
            self.preview = False

Let's try to use these configs in practice. Our demo `.bag` file was filmed in 1280x720 aspect ratio with 30FPS. We want to output a smaller version of that: a 640x480 aspect ratio with 30FPS.

In [80]:
intel_config = _INTEL_CONFIG_('20251116_155458-Color', w=1280, h=720, fps=30, type=rs.stream.color, format=rs.format.rgb8)
output_config = _OUTPUT_CONFIG_('20251116_155458-Color', w=640, h=480, fps=30, record=False, preview=True)

## Extracting Color Frames

Remember that each `.bag` file has its own set of streams. So we need some way to read each stream and output each stream.

In [81]:
from random import choice
from string import ascii_uppercase
import shutil

# Realsense primitives
_pipeline = rs.pipeline()
_config = rs.config()
_aligner = rs.align(rs.stream.color)

# We tell `_config` about the input bag and whether we want to repeat playback
rs.config.enable_device_from_file(_config, _INPUT_FILE, repeat_playback=False)

# We need to "enable" the depth and color stream separately. This example shows how to do that with colors
def set_stream(rc, ic):
    rc.enable_stream(ic.type, ic.width, ic.height, ic.format, ic.fps)

set_stream(_config, intel_config)

# The complicated matter is that when we read `.bag` data, it's not guarnateed that we'll be given frames in order
# So to fix this, we'll create a temp folder where we store all images extracted from the streams
def mkdirs(query_dir:str, delete_existing:bool=True):
    if delete_existing and os.path.exists(query_dir): 
        shutil.rmtree(query_dir)            # If the folder already exists, delete it
    os.makedirs(query_dir, exist_ok=True)   # Create a new empty directory
    return query_dir                        # Return the directory to indicate completion

def create_random_str(length:int = 12):
    return ''.join(choice(ascii_uppercase) for i in range(length))

temp_dir = mkdirs(create_random_str())

# With the streams configured and our temp directory created, let's start the pipeline
_pipeline.start(_config)

# Getting frames is purely to get the frames in image form, and save them in a cache
gc.disable()
frames_cache = []

# Continuously loop. Get all frames
while True:
    # Get frameset of color and depth
    frame_present, frames = _pipeline.try_wait_for_frames()
    if not frame_present:
        print("Frames no longer present!")
        break

    # Extract frame details
    aligned_frames = _aligner.process(frames)
    color_frame = aligned_frames.get_color_frame()
    if not color_frame:
        print("ERROR: color frame is not valid")
        continue
    
    # Get the current frame number
    frame_number = color_frame.get_frame_number()
    
    # Extract the image as an array
    try:
        color_image = np.asanyarray(color_frame.get_data())
    except Exception as error:
        print("ERROR: color image array extraction", error)
        break
            
     # Store frames in cache
    color_image_2 = color_image.copy()
    color_image_3 = cv2.resize(color_image_2, (output_config.width, output_config.height), output_config.interpolator)
    out_filename = os.path.join(temp_dir, f'{frame_number}.png')
    frames_cache.append([frame_number, out_filename])
    cv2.imwrite(out_filename, color_image_3)

# Upon exiting the loop, stop and release everything
gc.enable()
_pipeline.stop()


# Re-sort `frames_cache` based on the frame number
sorted_frames = sorted(frames_cache, key=lambda x: x[0])

_video_writer = cv2.VideoWriter(
    _OUTPUT_FILE, 
    cv2.VideoWriter_fourcc(*'MP4V'), 
    output_config.fps, 
    [output_config.width, output_config.height]
)

# Iterate through sorted frames
for frame in sorted_frames:
    img = cv2.imread(frame[1])
    if img is None:
        print(f"ERROR: Couldn't read image {frame[0]}")
        continue
    _video_writer.write(img)

# Release writer
_video_writer.release()

# Delete temp folder
shutil.rmtree(temp_dir)

RuntimeError: Couldn't resolve requests